In [1]:
# Setup
# %pip install -q openai python-dotenv


## 1) Import


In [2]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import json


## 2) Load environment variables - please read instructions carefully


In [3]:
# if you are running in local, uncomment below line. also make sure you shall have a .env file
# load_dotenv()

In [4]:
# if you are running in google colab, uncomment below line. and replace "Your_API_Key" with your own openAI API key
#os.environ["OPENAI_API_KEY"] = "Your_API_Key"

In [5]:
# llm_name = "gpt-4o"
# openai_key = os.getenv("OPENAI_API_KEY")
# client = OpenAI(api_key=openai_key)

client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"  # placeholder, Ollama ignores this
)

llm_name = "qwen3"

## 3) LLM Call

In [6]:
messages = []
message = {"role": "user", "content": "Plan a 2-day Tokyo trip for 2 adults. Mid comfort."}
messages.append(message)

In [7]:
# Create an agent using OpenAI native tool calling
response = client.chat.completions.create(
    model=llm_name,
    temperature=0.4,
    messages=messages
)

In [8]:
print("\n messages sent to LLM ")
print("\n", messages)


 messages sent to LLM 

 [{'role': 'user', 'content': 'Plan a 2-day Tokyo trip for 2 adults. Mid comfort.'}]


In [9]:
print("\n response from LLM ")
print("\n", response)


 response from LLM 

 ChatCompletion(id='chatcmpl-929', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='**2-Day Tokyo Itinerary for 2 Adults (Mid Comfort)**  \n*Balancing iconic landmarks, cultural experiences, and relaxed moments with mid-range comfort.*\n\n---\n\n### **Day 1: Traditional Meets Modern**  \n**Morning**  \n- **Asakusa & Senso-ji Temple (10:00 AM)**  \n  Start at Tokyo’s oldest temple, surrounded by traditional shops and the iconic **Kaminarimon Gate**. Stroll through the **Nakamise Street** for souvenirs and street food (try **tamagoyaki** or **yatsuhashi**).  \n- **Tokyo Skytree (1:00 PM)**  \n  Take a short train to **Shinjuku** and visit the **Tokyo Skytree** for panoramic views. Opt for the **Observation Deck** for a midday photo op.  \n\n**Afternoon**  \n- **Shibuya Crossing & Shopping (3:00 PM)**  \n  Walk across **Shibuya Crossing**, then explore **Shibuya Scramble Crossing** for a lively vibe. Head to **Shibu

In [10]:
assistant_message = response.choices[0].message
print(assistant_message.content)

**2-Day Tokyo Itinerary for 2 Adults (Mid Comfort)**  
*Balancing iconic landmarks, cultural experiences, and relaxed moments with mid-range comfort.*

---

### **Day 1: Traditional Meets Modern**  
**Morning**  
- **Asakusa & Senso-ji Temple (10:00 AM)**  
  Start at Tokyo’s oldest temple, surrounded by traditional shops and the iconic **Kaminarimon Gate**. Stroll through the **Nakamise Street** for souvenirs and street food (try **tamagoyaki** or **yatsuhashi**).  
- **Tokyo Skytree (1:00 PM)**  
  Take a short train to **Shinjuku** and visit the **Tokyo Skytree** for panoramic views. Opt for the **Observation Deck** for a midday photo op.  

**Afternoon**  
- **Shibuya Crossing & Shopping (3:00 PM)**  
  Walk across **Shibuya Crossing**, then explore **Shibuya Scramble Crossing** for a lively vibe. Head to **Shibuya Scramble** for lunch (try **ramen** or **katsu**).  
- **Shibuya Sky (5:00 PM)**  
  Relax at the **Shibuya Sky** rooftop bar for sunset views and a drink.  

**Evening*

## 4) LLM Call - Structured output (prompt engineering)
**issues:**
- not reliable
- Notice how much of the prompt is about formatting instead of solving the actual task.

In [11]:
messages = []
message = {"role": "user", "content": """Plan a 2-day Tokyo trip for 2 adults. Mid comfort. Output in JSON format, Return ONLY valid JSON.

The JSON must exactly follow this schema:
{
  "destination": string,
  "duration_days": integer,
  "travellers": integer,
  "comfort_level": string,
  "itinerary": [
    {
      "day": integer,
      "activities": [
        string
      ]
    }
  ]
}

Do not return markdown.
Do not wrap the JSON in ```json.
Do not include any additional text before or after the JSON."""
}
messages.append(message)

In [12]:
# Create an agent using OpenAI native tool calling
response = client.chat.completions.create(
    model=llm_name,
    temperature=0.4,
    messages=messages
)

In [13]:
assistant_message = response.choices[0].message
print(assistant_message.content)

{
  "destination": "Tokyo",
  "duration_days": 2,
  "travellers": 2,
  "comfort_level": "mid",
  "itinerary": [
    {
      "day": 1,
      "activities": [
        "Visit Tokyo Tower",
        "Explore Shibuya Crossing and shopping streets",
        "Lunch at a local ramen restaurant",
        "Stroll through Yoyogi Park",
        "Evening walk in Harajuku district"
      ]
    },
    {
      "day": 2,
      "activities": [
        "Tour Senso-ji Temple in Asakusa",
        "Visit Tokyo National Museum",
        "Lunch in Ueno Park area",
        "Shopping in Ginza district",
        "Dinner at a Shimokitazawa izakaya"
      ]
    }
  ]
}


## 5) LLM Call - Structured output (proper way)
- Returns valid structure.
- But still need to validate business rules.

In [14]:
messages = []
message = {"role": "user", "content": "Plan a 2-day Tokyo trip for 2 adults. Mid comfort."}
messages.append(message)

In [15]:
response = client.chat.completions.create(
    model=llm_name,
    temperature=0.4,
    messages=messages,
    response_format={
        "type": "json_schema",
        "json_schema": {
            "name": "trip_plan",
            "schema": {
                "type": "object",
                "properties": {
                    "destination": {
                        "type": "string"
                    },
                    "duration_days": {
                        "type": "integer"
                    },
                    "travellers": {
                        "type": "integer"
                    },
                    "comfort_level": {
                        "type": "string"
                    },
                    "itinerary": {
                        "type": "array",
                        "items": {
                            "type": "object",
                            "properties": {
                                "day": {
                                    "type": "integer"
                                },
                                "activities": {
                                    "type": "array",
                                    "items": {
                                        "type": "string"
                                    }
                                }
                            },
                            "required": [
                                "day",
                                "activities"
                            ]
                        }
                    }
                },
                "required": [
                    "destination",
                    "duration_days",
                    "travellers",
                    "comfort_level",
                    "itinerary"
                ]
            }
        }
    }
)

In [16]:
print("\nMessages sent to LLM")
print(messages)


Messages sent to LLM
[{'role': 'user', 'content': 'Plan a 2-day Tokyo trip for 2 adults. Mid comfort.'}]


In [17]:
print("\nResponse")
print(response)


Response
ChatCompletion(id='chatcmpl-467', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='{\n  "destination": "Tokyo, Japan",\n  "duration_days": 2,\n  "travellers": 2,\n  "comfort_level": "mid",\n  "itinerary": [\n    {\n      "day": 1,\n      "activities": [\n        "Morning: Explore **Asakusa** – Visit Senso-ji Temple, stroll through Nakamise Street, and enjoy local street food (like tamagoyaki and yakitori).",\n        "Midday: Head to **Ginza** – Shop at department stores (e.g., Mitsukoshi), dine at a mid-range izakaya or ramen shop.",\n        "Afternoon: Visit **Shibuya Crossing** and **Shibuya Scramble** – Capture iconic photos and explore nearby cafes or arcades.",\n        "Evening: Dine at a **ryokan** or traditional restaurant (e.g., Kappabashi Street for kaiseki or sushi) for a cultural experience."\n      ]\n    },\n    {\n      "day": 2,\n      "activities": [\n        "Morning: Discover **Akihabara** – Explore elec

In [18]:
assistant_message = response.choices[0].message
print(assistant_message.content)

{
  "destination": "Tokyo, Japan",
  "duration_days": 2,
  "travellers": 2,
  "comfort_level": "mid",
  "itinerary": [
    {
      "day": 1,
      "activities": [
        "Morning: Explore **Asakusa** – Visit Senso-ji Temple, stroll through Nakamise Street, and enjoy local street food (like tamagoyaki and yakitori).",
        "Midday: Head to **Ginza** – Shop at department stores (e.g., Mitsukoshi), dine at a mid-range izakaya or ramen shop.",
        "Afternoon: Visit **Shibuya Crossing** and **Shibuya Scramble** – Capture iconic photos and explore nearby cafes or arcades.",
        "Evening: Dine at a **ryokan** or traditional restaurant (e.g., Kappabashi Street for kaiseki or sushi) for a cultural experience."
      ]
    },
    {
      "day": 2,
      "activities": [
        "Morning: Discover **Akihabara** – Explore electronics shops, anime culture, and the **Tokyo Metropolitan Government Building** for panoramic views.",
        "Midday: Visit **Meiji Shrine** – Enjoy serene gard